# Running heavy work in the JupyDo job queue (Python)

Your environment has limited CPU and RAM. Heavy work goes to the **job queue**: put `%%job` on the
first line of a cell and the cell runs as a **batch job**, with the resources you ask for, when they are free.

How a job works:
- it runs in a **new container**, made from a snapshot of this environment taken when you submit it:
  same software (also what you installed yourself) and the same `/sharedFolder`;
- it runs the cell **alone**: variables of this notebook are **not** available in the job;
- only the **files it writes** survive: save your results in `/sharedFolder` (the job starts in this notebook's folder);
- `--time` is how long you expect it to run: if others are waiting, a job running longer is stopped.

Options: `--time` (e.g. `30m`, `2h`), `--cpus`, `--mem` (GB), `--gpus`, `--name`.

## 1. Queue status
`%jobs` lists your jobs and the free resources of the queue.

In [ ]:
%jobs

## 2. A first job
It prints a message, waits 30 seconds and ends. Run the cell: it returns at once with the job id.

In [ ]:
%%job --time 5m --cpus 1 --mem 2 --name hello
import socket, time
print('Hello from the job container', socket.gethostname())
time.sleep(30)
print('done')

## 3. Follow it
`%jobs` shows its state (pending, running, done...). `%job_log <id>` shows what it printed: replace `<id>` with the id printed above.

In [ ]:
%jobs

In [ ]:
# %job_log <id>

## 4. A job that saves a result
The job computes something and **writes it to a file**. When it is done, read the file here.

(Only the Python standard library is used, so it works in every image.)

In [ ]:
%%job --time 10m --cpus 2 --mem 4 --name squares
import json, os
total = sum(i * i for i in range(10_000_000))
os.makedirs('results', exist_ok=True)
with open('results/squares.json', 'w') as f:
    json.dump({'sum_of_squares': total}, f)
print('saved results/squares.json:', total)

In [ ]:
# when %jobs says 'done':
import json
with open('results/squares.json') as f:
    print(json.load(f))

## 5. Cancel a job
`%job_cancel <id>` cancels a pending job or stops a running one.

In [ ]:
# %job_cancel <id>

## GPU jobs
If the server has GPUs, ask for them with `--gpus`: only the job sees them.

In [ ]:
# %%job --time 1h --cpus 4 --mem 16 --gpus 1 --name gpu_test
# import subprocess
# print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)